# Prepare Local FiftyOne Dataset + Embedding Cache

This notebook is the one-time preparation step for all experiment notebooks.

It performs:
1. Load the Hugging Face FiftyOne export into a **persistent local FiftyOne dataset** via `fiftyone.utils.huggingface.load_from_hub`
2. Precompute and store patch embeddings for all backbone models used in experiments

After this, configure experiments with:
- `config.dataset.source = "fiftyone"`
- `config.dataset.fo_dataset_name = LOCAL_FO_DATASET_NAME`
- `config.dataset.fo_patches_field = "sam3_segmentations"`
- `config.dataset.fo_split_field = "closed_set_split"`
- `config.dataset.fo_label_field = "ground_truth"`

In [1]:
import sys
from pathlib import Path

# Project path setup
project_root = Path.cwd().parent / "camera-trap-footage"
sys.path.insert(0, str(project_root / "src"))

import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub
from huggingface_hub import snapshot_download

from jaguars.ingestion.processing.add_multi_backbone_embeddings import run_processing as precache_embeddings

HF_REPO = "JaguarCameraTrap/jaguars_camera_trap_0226-segmented_deduplicated"
LOCAL_FO_DATASET_NAME = "JID_HF_0226_Segmented_Deduplicated_Cached"
LOCAL_HF_SNAPSHOT_DIR = project_root / "data" / "intermediate" / "hf_cache" / "jaguars_camera_trap_0226_segmented_deduplicated"

# Controls
OVERWRITE_LOCAL_DATASET = False
PERSISTENT = True
NUM_DOWNLOAD_WORKERS = 8
LOAD_BATCH_SIZE = 64

# Embedding cache controls
PRECOMPUTE_EMBEDDINGS = True
PRECOMPUTE_BATCH_SIZE = 16
PRECOMPUTE_OVERWRITE = False

print(f"HF repo: {HF_REPO}")
print(f"Local FiftyOne dataset: {LOCAL_FO_DATASET_NAME}")
print(f"Local snapshot dir: {LOCAL_HF_SNAPSHOT_DIR}")
print(f"Overwrite local dataset: {OVERWRITE_LOCAL_DATASET}")
print(f"Precompute embeddings: {PRECOMPUTE_EMBEDDINGS}")

HF repo: JaguarCameraTrap/jaguars_camera_trap_0226-segmented_deduplicated
Local FiftyOne dataset: JID_HF_0226_Segmented_Deduplicated_Cached
Local snapshot dir: /sc/home/philipp.kolbe/JID/camera-trap-footage/camera-trap-footage/data/intermediate/hf_cache/jaguars_camera_trap_0226_segmented_deduplicated
Overwrite local dataset: False
Precompute embeddings: True


In [2]:
# 1) Load HF dataset into persistent local FiftyOne dataset
if fo.dataset_exists(LOCAL_FO_DATASET_NAME) and OVERWRITE_LOCAL_DATASET:
    print(f"Deleting existing dataset: {LOCAL_FO_DATASET_NAME}")
    fo.delete_dataset(LOCAL_FO_DATASET_NAME)

if fo.dataset_exists(LOCAL_FO_DATASET_NAME) and not OVERWRITE_LOCAL_DATASET:
    dataset = fo.load_dataset(LOCAL_FO_DATASET_NAME)
    print(f"✓ Using existing local dataset: {LOCAL_FO_DATASET_NAME}")
else:
    dataset = None
    print("Trying FiftyOne Hub loader...")
    try:
        dataset = load_from_hub(
            repo_id=HF_REPO,
            name=LOCAL_FO_DATASET_NAME,
            overwrite=OVERWRITE_LOCAL_DATASET,
            persistent=PERSISTENT,
            batch_size=LOAD_BATCH_SIZE,
            num_workers=NUM_DOWNLOAD_WORKERS,
        )
        print(f"✓ Loaded via load_from_hub into: {LOCAL_FO_DATASET_NAME}")
    except Exception as e:
        print(f"⚠ load_from_hub failed: {e}")
        print("Falling back to raw snapshot -> FiftyOneDataset import...")

        LOCAL_HF_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
        snapshot_dir = snapshot_download(
            repo_id=HF_REPO,
            repo_type="dataset",
            local_dir=str(LOCAL_HF_SNAPSHOT_DIR),
            local_dir_use_symlinks=False,
            resume_download=True,
        )
        snapshot_path = Path(snapshot_dir)

        candidate_roots = [snapshot_path]
        candidate_roots.extend([p for p in snapshot_path.iterdir() if p.is_dir()])

        export_root = None
        for root in candidate_roots:
            if (root / "metadata.json").exists() and (root / "samples.json").exists():
                export_root = root
                break

        if export_root is None:
            raise ValueError(
                "Could not find FiftyOne export files (metadata.json + samples.json) in downloaded snapshot"
            )

        dataset = fo.Dataset.from_dir(
            dataset_dir=str(export_root),
            dataset_type=fo.types.FiftyOneDataset,
            name=LOCAL_FO_DATASET_NAME,
            persistent=PERSISTENT,
        )
        print(f"✓ Imported from snapshot export dir: {export_root}")

images_view = dataset.select_group_slices("image") if dataset.group_field else dataset
print(f"Total samples: {len(dataset)}")
print(f"Image samples: {len(images_view)}")
print(f"Group field: {dataset.group_field}")
split_field = "closed_set_split" if "closed_set_split" in images_view.get_field_schema() else "split"
print(f"Split field used for counts: {split_field}")
print(f"Split counts: {images_view.count_values(split_field)}")

✓ Using existing local dataset: JID_HF_0226_Segmented_Deduplicated_Cached
Total samples: 3590
Image samples: 3590
Group field: group
Split field used for counts: closed_set_split
Split counts: {'train': 2928, 'val': 315, 'test': 347}


In [3]:
# 2) Precompute embeddings for all experiment backbones (robust, per-backbone)
BACKBONE_CACHE_SPECS = [
    {"name": "vit_large_patch16_dinov3.lvd1689m", "embedding_dim": 1024, "input_size": 512, "field_suffix": "DINOv3_Large"},
    {"name": "vit_base_patch16_dinov3.lvd1689m", "embedding_dim": 768, "input_size": 512, "field_suffix": "DINOv3_Base"},
    {"name": "conservationxlabs/miewid-msv2", "embedding_dim": 2152, "input_size": 440, "field_suffix": "MiewID_MSv2"},
    {"name": "conservationxlabs/miewid-msv3", "embedding_dim": 2152, "input_size": 440, "field_suffix": "MiewID_MSv3"},
    {"name": "vit_large_patch14_dinov2.lvd142m", "embedding_dim": 1024, "input_size": 518, "field_suffix": "DINOv2_Large"},
    {"name": "vit_base_patch14_dinov2.lvd142m", "embedding_dim": 768, "input_size": 518, "field_suffix": "DINOv2_Base"},
    {"name": "vit_small_patch14_dinov2.lvd142m", "embedding_dim": 384, "input_size": 518, "field_suffix": "DINOv2_Small"},
    {"name": "hf-hub:BVRA/MegaDescriptor-L-384", "embedding_dim": 1536, "input_size": 384, "field_suffix": "BVRA_MegaDescriptor_L_384"},
    {"name": "hf-hub:BVRA/MegaDescriptor-B-224", "embedding_dim": 768, "input_size": 224, "field_suffix": "BVRA_MegaDescriptor_B_224"},
    {"name": "resnet50", "embedding_dim": 2048, "input_size": 224, "field_suffix": "ResNet50"},
    {"name": "convnext_base", "embedding_dim": 1024, "input_size": 224, "field_suffix": "ConvNeXt_Base"},
    {"name": "convnextv2_base.fcmae_ft_in22k_in1k", "embedding_dim": 1024, "input_size": 224, "field_suffix": "ConvNeXtV2_Base"},
    {"name": "efficientnet_b3", "embedding_dim": 1536, "input_size": 288, "field_suffix": "EfficientNet_B3"},
    {"name": "hf-hub:timm/efficientnetv2_rw_m.agc_in1k", "embedding_dim": 2152, "input_size": 320, "field_suffix": "EfficientNetV2_RW_M"},
]

# Controls for robust caching
SKIP_MIEWID = False
STOP_ON_FIRST_ERROR = False

if PRECOMPUTE_EMBEDDINGS:
    selected_specs = []
    for spec in BACKBONE_CACHE_SPECS:
        if SKIP_MIEWID and spec["name"].startswith("conservationxlabs/miewid-"):
            continue
        selected_specs.append(spec)

    print(f"Preparing embeddings for {len(selected_specs)} backbones")
    cache_results = {}

    for idx, spec in enumerate(selected_specs, 1):
        backbone_name = spec["name"]
        print("=" * 90)
        print(f"[{idx}/{len(selected_specs)}] {backbone_name}")

        try:
            result = precache_embeddings(
                dataset_name=LOCAL_FO_DATASET_NAME,
                backbones=[spec],
                patches_field="sam3_segmentations",
                batch_size=PRECOMPUTE_BATCH_SIZE,
                overwrite=PRECOMPUTE_OVERWRITE,
                verbose=True,
            )
            cache_results[backbone_name] = result
            print(f"✓ Done: {backbone_name}")
        except Exception as e:
            cache_results[backbone_name] = {"status": "failed", "error": str(e)}
            print(f"✗ Failed: {backbone_name}")
            print(f"  Error: {e}")
            if STOP_ON_FIRST_ERROR:
                raise

    print("\n✓ Embedding precompute finished")
    failed = [k for k, v in cache_results.items() if isinstance(v, dict) and v.get("status") == "failed"]
    print(f"Completed: {len(cache_results) - len(failed)} | Failed: {len(failed)}")
    if failed:
        print("Failed backbones:")
        for name in failed:
            print(f"  - {name}: {cache_results[name]['error']}")
else:
    print("Skipping embedding precompute")

Preparing embeddings for 14 backbones
[1/14] vit_large_patch16_dinov3.lvd1689m
07:53:25 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - Computing embeddings for 1 backbones
07:53:25 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   - vit_large_patch16_dinov3.lvd1689m → embeddings_DINOv3_Large
07:53:25 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - Resource validation passed
07:53:25 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - Using device: cuda
07:53:25 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - 
07:53:25 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - Processing backbone: vit_large_patch16_dinov3.lvd1689m
07:53:25 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO -   Embedding field: embeddings_DINOv3_Large
07:53:25 - jid_logger.ingestion.processing.add_multi_backbone_embeddings - INFO - ===================

In [4]:
# 3) Verify embedding fields exist (detection-level for patch embeddings) and print config snippet
dataset = fo.load_dataset(LOCAL_FO_DATASET_NAME)
images_view = dataset.select_group_slices("image") if dataset.group_field else dataset

required_fields = [f"embeddings_{spec['field_suffix']}" for spec in BACKBONE_CACHE_SPECS]
patches_field = "sam3_segmentations"

# Collect schema information
sample_schema = images_view.get_field_schema()
flat_schema = images_view.get_field_schema(flat=True)

print(f"Dataset: {LOCAL_FO_DATASET_NAME}")
print(f"Image samples: {len(images_view)}")
print(f"Patches field present: {patches_field in sample_schema}")

# Detection-level existence check
existing_fields = []
missing_fields = []
field_stats = {}

if patches_field in sample_schema:
    # Find one sample with at least one detection for fast field introspection
    first_detection = None
    for sample in images_view:
        patches = getattr(sample, patches_field, None)
        if patches is not None and getattr(patches, "detections", None):
            if len(patches.detections) > 0:
                first_detection = patches.detections[0]
                break

    for field in required_fields:
        # field may be represented in flat schema or as detection attribute
        nested_field = f"{patches_field}.detections.{field}"
        present_in_flat = nested_field in flat_schema
        present_on_detection = first_detection is not None and hasattr(first_detection, field)

        if present_in_flat or present_on_detection:
            existing_fields.append(field)
            # Count number of detections containing this field
            detection_count = 0
            for sample in images_view:
                patches = getattr(sample, patches_field, None)
                if patches is None or getattr(patches, "detections", None) is None:
                    continue
                for det in patches.detections:
                    if hasattr(det, field) and getattr(det, field) is not None:
                        detection_count += 1
            field_stats[field] = detection_count
        else:
            missing_fields.append(field)
            field_stats[field] = 0
else:
    missing_fields = list(required_fields)
    for field in required_fields:
        field_stats[field] = 0

print(f"\nExisting detection embedding fields ({len(existing_fields)}):")
for field in existing_fields:
    print(f"  ✓ {field} (detections with embedding: {field_stats[field]})")

if missing_fields:
    print(f"\nMissing detection embedding fields ({len(missing_fields)}):")
    for field in missing_fields:
        print(f"  - {field}")

print("\nUse this in experiment notebooks:")
print("config.dataset.source = 'fiftyone'")
print(f"config.dataset.fo_dataset_name = '{LOCAL_FO_DATASET_NAME}'")
print("config.dataset.fo_patches_field = 'sam3_segmentations'")
print("config.dataset.fo_split_field = 'closed_set_split'")
print("config.dataset.fo_label_field = 'ground_truth'")
print("config.dataset.fo_embeddings_field = None  # backbone-specific auto-detection/cached field")

Dataset: JID_HF_0226_Segmented_Deduplicated_Cached
Image samples: 3590
Patches field present: True

Existing detection embedding fields (14):
  ✓ embeddings_DINOv3_Large (detections with embedding: 2789)
  ✓ embeddings_DINOv3_Base (detections with embedding: 2789)
  ✓ embeddings_MiewID_MSv2 (detections with embedding: 2789)
  ✓ embeddings_MiewID_MSv3 (detections with embedding: 2789)
  ✓ embeddings_DINOv2_Large (detections with embedding: 2789)
  ✓ embeddings_DINOv2_Base (detections with embedding: 2789)
  ✓ embeddings_DINOv2_Small (detections with embedding: 2789)
  ✓ embeddings_BVRA_MegaDescriptor_L_384 (detections with embedding: 2789)
  ✓ embeddings_BVRA_MegaDescriptor_B_224 (detections with embedding: 2789)
  ✓ embeddings_ResNet50 (detections with embedding: 2789)
  ✓ embeddings_ConvNeXt_Base (detections with embedding: 2789)
  ✓ embeddings_ConvNeXtV2_Base (detections with embedding: 2789)
  ✓ embeddings_EfficientNet_B3 (detections with embedding: 2789)
  ✓ embeddings_EfficientNet